### Computing ESM-2 embeddings with internet access:

In [2]:
import torch
import esm # install via pip install fair-esm
import numpy as np

#when you call this model the following function for the first time, this will download the model from the internet
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval();  

device =  torch.device("cuda" if torch.cuda.is_available() else "cpu")

#This function takes a protein sequence and returns the mean-embedding of the protein
def get_esm_embedding(model, alphabet, sequence):
    sequences = [("P", sequence)]
    batch_labels, batch_strs, batch_tokens = batch_converter(sequences)
    with torch.no_grad():
        results = model(batch_tokens.to(device), repr_layers=[6], return_contacts=False)
    token_representations = results["representations"][6].cpu().numpy()
    #we don't want to use the first and last token:
    protein_representations = np.mean(token_representations[0,1:-1,:], axis=0)
    return protein_representations

#example
sequence = "MAGLQKQK"
embedding = get_esm_embedding(model, alphabet, sequence)
print(embedding.shape)

(320,)


In [6]:
sequence = "M"*1025
embedding = get_esm_embedding(model, alphabet, sequence)
print(embedding.shape)

(320,)


In [2]:
embedding

array([ 7.19903857e-02, -3.47023636e-01,  3.14934403e-01,  2.82230228e-01,
       -1.34158283e-01, -1.82931587e-01, -2.74978608e-01,  2.45038420e-03,
       -1.58777282e-01, -7.24265575e-02, -2.89227158e-01,  3.20510060e-01,
       -5.72140329e-03,  2.04135180e-01, -2.80416012e-01,  3.95817570e-02,
        9.96295288e-02, -9.36608315e-02,  2.08584651e-01, -2.11283803e-01,
       -4.22989398e-01,  8.63111988e-02, -1.13524765e-01,  1.26650766e-01,
        5.40021658e-02,  3.05889457e-01,  3.68483514e-02,  7.57430941e-02,
        1.11543186e-01, -1.20182380e-01, -2.66182888e-03, -1.27431348e-01,
        1.14369988e-01, -3.25008899e-01,  3.15570474e-01, -4.69514012e-01,
        1.71687752e-01, -1.86191127e-01,  3.76058370e-01,  1.89027578e-01,
       -2.45532542e-01,  2.31385440e-01,  1.81106761e-01,  5.02078943e-02,
       -2.19936669e-01, -5.68872020e-02, -1.31243920e+00,  1.44530594e-01,
       -1.47734493e-01, -4.11876403e-02,  9.28843021e-02,  5.22227511e-02,
        2.47631773e-01,  

### Computing ESM-2 emebddings without internet access (on HPC):

You first need to download the model paramaters of the ESM-2 model ("esm2_t6_8M_UR50D-contact-regression.pt", "esm2_t6_8M_UR50D.pt") from data/worksheet4. You need to store those files on the HPC. The following code shows you how to load the model that is stored on a local file without downloading it from the internet:

In [ ]:
model, alphabet = esm.pretrained.load_model_and_alphabet_local(model_location = "/path_to_parameter_weights/esm2_t6_8M_UR50D.pt")
batch_converter = alphabet.get_batch_converter()
model.eval();

In [ ]:
#The following code is identical to the code above for the downloaded model:

In [ ]:
device =  torch.device("cuda" if torch.cuda.is_available() else "cpu")
#This function takes a protein sequence and returns the mean-embedding of the protein
def get_esm_embedding(model, alphabet, sequence):
    sequences = [("P", sequence)]
    batch_labels, batch_strs, batch_tokens = batch_converter(sequences)
    with torch.no_grad():
        results = model(batch_tokens.to(device), repr_layers=[6], return_contacts=False)
    token_representations = results["representations"][6].cpu().numpy()
    #we don't want to use the first and last token:
    protein_representations = np.mean(token_representations[0,1:-1,:], axis=0)
    return protein_representations


embedding = get_esm_embedding(model, alphabet, sequence)
print(embedding.shape)